In [5]:
# Run once
%pip install pgmpy kaggle pandas scikit-learn matplotlib seaborn pyarrow

Note: you may need to restart the kernel to use updated packages.


In [6]:
# run once, goes to ftp.ncdc.noaa.gov/pub/data/gsod/2002 and grabs data
from ftplib import FTP
import os

os.makedirs("../data/raw/gsod", exist_ok=True)
local_tar = "../data/raw/gsod/gsod_2002.tar"

if not os.path.exists(local_tar):
    print("Working...")
    ftp = FTP("ftp.ncdc.noaa.gov")
    ftp.login("ftp", "gavin.burchett@wsu.edu")   
    ftp.cwd("/pub/data/gsod/2002")
    print("Files:", ftp.nlst()[:5])               

    with open(local_tar, "wb") as f:
        ftp.retrbinary("RETR gsod_2002.tar", f.write)
    ftp.quit()
    print(f"Downloaded → {local_tar}")
else:
    print("Already downloaded")

Already downloaded


In [7]:
# Extract the tar file
import tarfile

extract_dir = "../data/raw/gsod/2002"
os.makedirs(extract_dir, exist_ok=True)

with tarfile.open(local_tar, "r") as tar:
    members = tar.getnames()
    tar.extractall(extract_dir)
    print(f"Extracted {len(members)} station files")
    print("Sample:", members[:3])

KeyboardInterrupt: 

In [ ]:
# Read the .op.gz files into a single DataFrame
import pandas as pd
import glob
import gzip

COL_NAMES = [
    "STN", "WBAN", "YEARMODA",
    "TEMP", "TEMP_CNT", "DEWP", "DEWP_CNT",
    "SLP",  "SLP_CNT",  "STP",  "STP_CNT",
    "VISIB","VISIB_CNT","WDSP", "WDSP_CNT",
    "MXSPD","GUST","MAX","MIN","PRCP","SNDP","FRSHTT"
]

# NOAA missing-value sentinels per element
SENTINELS = {
    "TEMP": 9999.9, "DEWP": 9999.9, "SLP": 9999.9, "STP": 9999.9,
    "VISIB": 999.9, "WDSP": 999.9,  "MXSPD": 999.9, "GUST": 9999.9,
    "MAX": 9999.9,  "MIN": 9999.9,  "PRCP": 99.99,   "SNDP": 999.9,
}

def read_gsod_file(filepath):
    with gzip.open(filepath, "rt") as f:
        df = pd.read_csv(f, sep=r"\s+", skiprows=1, header=None, names=COL_NAMES)

    # MAX and MIN may carry a trailing '*' flag indicating a provisional value
    df["MAX"] = pd.to_numeric(
        df["MAX"].astype(str).str.replace("*", "", regex=False), errors="coerce"
    )
    df["MIN"] = pd.to_numeric(
        df["MIN"].astype(str).str.replace("*", "", regex=False), errors="coerce"
    )
    # PRCP carries a trailing letter flag (A–I indicating accumulation window)
    df["PRCP"] = pd.to_numeric(
        df["PRCP"].astype(str).str.replace(r"[A-Za-z]", "", regex=True), errors="coerce"
    )

    # Apply missing sentinels
    for col, sentinel in SENTINELS.items():
        df[col] = df[col].replace(sentinel, pd.NA)

    return df

gz_files = glob.glob(f"{extract_dir}/*.op.gz")
print(f"Found {len(gz_files)} station files")

df_raw = pd.concat([read_gsod_file(f) for f in gz_files], ignore_index=True)
print(f"Total records: {len(df_raw):,}")
df_raw.head()

Found 8990 station files


KeyboardInterrupt: 

In [ ]:
# Parse YEARMODA → proper date
df_raw["DATE"] = pd.to_datetime(df_raw["YEARMODA"].astype(str), format="%Y%m%d")

# FRSHTT indicates the following Fog | Rain | Snow | Hail | Thunder | Tornado
df_raw["FRSHTT"]  = df_raw["FRSHTT"].astype(str).str.zfill(6)
df_raw["FOG"]     = df_raw["FRSHTT"].str[0].astype(int)
df_raw["RAIN"]    = df_raw["FRSHTT"].str[1].astype(int)
df_raw["SNOW"]    = df_raw["FRSHTT"].str[2].astype(int)
df_raw["HAIL"]    = df_raw["FRSHTT"].str[3].astype(int)
df_raw["THUNDER"] = df_raw["FRSHTT"].str[4].astype(int)
df_raw["TORNADO"] = df_raw["FRSHTT"].str[5].astype(int)

print(df_raw.dtypes)
print(df_raw[["DATE","TEMP","DEWP","SLP","VISIB","WDSP","MXSPD","GUST",
               "MAX","MIN","PRCP","SNDP","RAIN","SNOW","FOG"]].describe())

In [ ]:
# Save the cleaned DataFrame to CSV
# Need to check for handling missing vals and some more cleaning, some format changes maybe
# Need to check more on what Bayes networks want for data format and input types
# have not checked outliers yet either
# need station filtering maybe? depends on the route we want to go.

os.makedirs("../data/processed", exist_ok=True)
df_raw.to_csv("../data/processed/gsod_2002.csv", index=False)
print("Saved → ../data/processed/gsod_2002.csv")

need a network structure

- pressure and temperature are more "fundamental" factors
- precipitation will be rain and snow, temp and pressure related
- wind will be pressure and temperatre impacted, not by precip
- dewpoint - idk, need to learn more weather
- fog plays in ?????
- for Bayes there should be binning for each type but need classifiers?
- pressure = high/low
- temp = cold, avg, warm
- wind = none, low, high ???
- dew point ??
- fog, rain, snow can either be bools or maybe light/heavy/none classifications?    For the precipitations have precip be light/heavy then can just be bools for the type of precip? fog is still ??
- 

Meeting Notes:
double check for provided bayesian graph information, otherwise look into other sources for graph templates. 
If we can't find the information ask the TA for assistance

take a look at a simplified dynamic bayesian network through another model. using t-1 to see hsitory, this may simplify our process.
^ The hidden markov model is what may be good here.